# Remove Chain-of-Thought (CoT) Feedbacks

This notebook detects and removes feedbacks containing internal reasoning/CoT patterns that pollute the training signal.

## Problematic patterns to detect:
- "Is your feedback neutral ? Yes your feedback is neutral..."
- "Does your feedback allow..."
- Meta-commentary about the feedback itself
- Self-questioning patterns
- Internal reasoning artifacts

## 1. Setup and Imports

In [1]:
import pandas as pd
import json
import re
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries loaded successfully!")

✓ Libraries loaded successfully!


## 2. Load Cleaned Dataset

In [2]:
print("="*100)
print("LOADING DATASET")
print("="*100)
print()

dataset_path = Path('../data/cleaned_dataset.csv')

if not dataset_path.exists():
    print(f"✗ Dataset not found at: {dataset_path}")
    print("  Please run the data_cleaning.ipynb notebook first!")
else:
    df = pd.read_csv(dataset_path)
    print(f"✓ Loaded {len(df):,} samples")
    print(f"\nColumns: {df.columns.tolist()}")

LOADING DATASET

✓ Loaded 11,898 samples

Columns: ['code_id', 'author_id', 'code_snippet', 'generated_feedback']


## 3. Define CoT Detection Patterns

In [3]:
# Define patterns that indicate CoT/meta-commentary
COT_PATTERNS = [
    # Self-questioning patterns
    r'Is your feedback',
    r'Does your feedback',
    r'Is the feedback',
    r'Does the feedback',
    
    # Self-answering patterns
    r'Yes[,\s]+your feedback is',
    r'Yes[,\s]+the feedback is',
    r'No[,\s]+your feedback',
    r'No[,\s]+the feedback',
    
    # Meta-commentary about neutrality
    r'neutral and do not contain',
    r'does not contain any positive or negative',
    r'without positive or negative connotation',
    
    # Meta-commentary about student identification
    r'allow the student to rapidly',
    r'allows the student to',
    r'help the student identify',
    r'helps the student',
    
    # Internal reasoning markers
    r'Let me check',
    r'Let\'s verify',
    r'I need to',
    r'I should',
    r'I will',
    
    # Thinking out loud
    r'First[,\s]+I',
    r'Then[,\s]+I',
    r'Now[,\s]+I',
    
    # Self-evaluation
    r'This feedback',
    r'My feedback',
    r'The above feedback',
]

print("="*100)
print("COT DETECTION PATTERNS")
print("="*100)
print()
print(f"Total patterns defined: {len(COT_PATTERNS)}")
print()
print("Patterns:")
for i, pattern in enumerate(COT_PATTERNS, 1):
    print(f"  {i:2d}. {pattern}")

COT DETECTION PATTERNS

Total patterns defined: 26

Patterns:
   1. Is your feedback
   2. Does your feedback
   3. Is the feedback
   4. Does the feedback
   5. Yes[,\s]+your feedback is
   6. Yes[,\s]+the feedback is
   7. No[,\s]+your feedback
   8. No[,\s]+the feedback
   9. neutral and do not contain
  10. does not contain any positive or negative
  11. without positive or negative connotation
  12. allow the student to rapidly
  13. allows the student to
  14. help the student identify
  15. helps the student
  16. Let me check
  17. Let\'s verify
  18. I need to
  19. I should
  20. I will
  21. First[,\s]+I
  22. Then[,\s]+I
  23. Now[,\s]+I
  24. This feedback
  25. My feedback
  26. The above feedback


## 4. Detect CoT Feedbacks

In [4]:
def contains_cot(feedback):
    """Check if feedback contains CoT patterns."""
    if pd.isna(feedback):
        return False
    
    feedback_str = str(feedback)
    
    for pattern in COT_PATTERNS:
        if re.search(pattern, feedback_str, re.IGNORECASE):
            return True
    
    return False

def get_matching_patterns(feedback):
    """Get all patterns that match in the feedback."""
    if pd.isna(feedback):
        return []
    
    feedback_str = str(feedback)
    matching = []
    
    for pattern in COT_PATTERNS:
        if re.search(pattern, feedback_str, re.IGNORECASE):
            matching.append(pattern)
    
    return matching

print("="*100)
print("DETECTING COT FEEDBACKS")
print("="*100)
print()

# Apply detection
df['contains_cot'] = df['generated_feedback'].apply(contains_cot)
df['matching_patterns'] = df['generated_feedback'].apply(get_matching_patterns)

cot_count = df['contains_cot'].sum()
cot_percentage = (cot_count / len(df)) * 100

print(f"Total feedbacks: {len(df):,}")
print(f"Feedbacks with CoT: {cot_count:,} ({cot_percentage:.2f}%)")
print(f"Clean feedbacks: {len(df) - cot_count:,} ({100 - cot_percentage:.2f}%)")

DETECTING COT FEEDBACKS

Total feedbacks: 11,898
Feedbacks with CoT: 92 (0.77%)
Clean feedbacks: 11,806 (99.23%)


## 5. Analyze CoT Patterns

In [5]:
print("="*100)
print("PATTERN FREQUENCY ANALYSIS")
print("="*100)
print()

# Count pattern occurrences
pattern_counts = {}
for patterns in df['matching_patterns']:
    for pattern in patterns:
        pattern_counts[pattern] = pattern_counts.get(pattern, 0) + 1

# Sort by frequency
sorted_patterns = sorted(pattern_counts.items(), key=lambda x: x[1], reverse=True)

print("Most common CoT patterns:")
print()
for pattern, count in sorted_patterns[:20]:
    percentage = (count / len(df)) * 100
    print(f"  {count:5,} ({percentage:5.2f}%) - {pattern}")

PATTERN FREQUENCY ANALYSIS

Most common CoT patterns:

     44 ( 0.37%) - allow the student to rapidly
     36 ( 0.30%) - First[,\s]+I
     28 ( 0.24%) - Does your feedback
     17 ( 0.14%) - Yes[,\s]+the feedback is
     10 ( 0.08%) - This feedback
     10 ( 0.08%) - My feedback
      7 ( 0.06%) - Is your feedback
      4 ( 0.03%) - Yes[,\s]+your feedback is
      4 ( 0.03%) - allows the student to
      3 ( 0.03%) - Is the feedback
      2 ( 0.02%) - No[,\s]+the feedback
      2 ( 0.02%) - does not contain any positive or negative
      1 ( 0.01%) - neutral and do not contain
      1 ( 0.01%) - Then[,\s]+I
      1 ( 0.01%) - Does the feedback
      1 ( 0.01%) - Now[,\s]+I


## 6. Show Examples of CoT Feedbacks

In [6]:
print("="*100)
print("EXAMPLES OF COT FEEDBACKS")
print("="*100)
print()

# Get samples with CoT
cot_samples = df[df['contains_cot']].head(5)

for idx, (i, row) in enumerate(cot_samples.iterrows(), 1):
    print(f"Example {idx}:")
    print("-" * 100)
    print(f"\nMatching patterns: {', '.join(row['matching_patterns'])}")
    print(f"\nFeedback:")
    print(row['generated_feedback'][:500] + ("..." if len(row['generated_feedback']) > 500 else ""))
    print("\n" + "="*100 + "\n")

EXAMPLES OF COT FEEDBACKS

Example 1:
----------------------------------------------------------------------------------------------------

Matching patterns: Is your feedback, Does your feedback, Yes[,\s]+your feedback is, neutral and do not contain, allow the student to rapidly

Feedback:
The logical condition that checks for values that are too large does not account for all potential overflows.

Is your feedback neutral ? Yes your feedback is neutral and do not contain any positive or negative connotation
Does your feedback allow the student to rapidly identifiy his bug ? Yes your feedback identify a specific part of the code and allow the student to look for it
Does your feedback anonymize any function or variable name ? No it does not anonymize any part of the code
Does your ...


Example 2:
----------------------------------------------------------------------------------------------------

Matching patterns: First[,\s]+I

Feedback:
The loop incorrectly returns after the first i

## 7. Show Examples of Clean Feedbacks

In [7]:
print("="*100)
print("EXAMPLES OF CLEAN FEEDBACKS (No CoT)")
print("="*100)
print()

# Get samples without CoT
clean_samples = df[~df['contains_cot']].sample(5)

for idx, (i, row) in enumerate(clean_samples.iterrows(), 1):
    print(f"Example {idx}:")
    print("-" * 100)
    print(f"\nCode (first 200 chars):")
    print(row['code_snippet'][:200] + "...")
    print(f"\nFeedback:")
    print(row['generated_feedback'][:400] + ("..." if len(row['generated_feedback']) > 400 else ""))
    print("\n" + "="*100 + "\n")

EXAMPLES OF CLEAN FEEDBACKS (No CoT)

Example 1:
----------------------------------------------------------------------------------------------------

Code (first 200 chars):
char *my_strstr(char *str, char const *to_find)
{
    if (!str[0])
        return 0;
    for (int i = 0; to_find[i]; i++) {
        if (to_find[i] != str[i])
            return (my_strstr(str + 1, to_...

Feedback:
Your function may not handle cases where the substring appears after a non-matching prefix correctly. Consider how the function should behave when the start of the main string doesn't match the substring but the substring appears later.


Example 2:
----------------------------------------------------------------------------------------------------

Code (first 200 chars):
int my_compute_power_it(int nb, int p)
{
    int result = 1;

    if (p < 0 || p > 12) {
        return 0;
    } else if (p == 0) {
        return 1;
    } else {
        for (int i = 1; i <= p; i++) ...

Feedback:
The code does not h

## 8. Remove CoT Feedbacks

In [8]:
print("="*100)
print("REMOVING COT FEEDBACKS")
print("="*100)
print()

initial_count = len(df)
cot_count_before = df['contains_cot'].sum()

print(f"Before removal:")
print(f"  Total samples: {initial_count:,}")
print(f"  CoT feedbacks: {cot_count_before:,}")
print(f"  Clean feedbacks: {initial_count - cot_count_before:,}")
print()

# Remove CoT feedbacks
df_clean = df[~df['contains_cot']].copy()

# Drop the detection columns
df_clean = df_clean.drop(columns=['contains_cot', 'matching_patterns'])

removed_count = initial_count - len(df_clean)

print(f"After removal:")
print(f"  Total samples: {len(df_clean):,}")
print(f"  Removed: {removed_count:,} ({removed_count/initial_count*100:.2f}%)")
print()

print("✓ CoT feedbacks removed successfully!")

REMOVING COT FEEDBACKS

Before removal:
  Total samples: 11,898
  CoT feedbacks: 92
  Clean feedbacks: 11,806

After removal:
  Total samples: 11,806
  Removed: 92 (0.77%)

✓ CoT feedbacks removed successfully!


## 9. Validate Clean Dataset

In [9]:
print("="*100)
print("VALIDATING CLEAN DATASET")
print("="*100)
print()

# Re-check for CoT patterns
validation_check = df_clean['generated_feedback'].apply(contains_cot).sum()

if validation_check == 0:
    print("✓ Validation passed: No CoT patterns detected in clean dataset")
else:
    print(f"⚠ Warning: {validation_check} feedbacks still contain CoT patterns")

print()
print("Clean dataset statistics:")
print(f"  Total samples: {len(df_clean):,}")
print(f"  Unique code_ids: {df_clean['code_id'].nunique():,}")
print(f"  Unique authors: {df_clean['author_id'].nunique():,}")
print(f"  Unique feedbacks: {df_clean['generated_feedback'].nunique():,}")
print()

# Check for nulls
nulls = df_clean.isnull().sum()
if nulls.sum() > 0:
    print("Null values:")
    print(nulls[nulls > 0])
else:
    print("✓ No null values found")

VALIDATING CLEAN DATASET

✓ Validation passed: No CoT patterns detected in clean dataset

Clean dataset statistics:
  Total samples: 11,806
  Unique code_ids: 11,806
  Unique authors: 1,320
  Unique feedbacks: 11,061

✓ No null values found


## 10. Compare Before/After Statistics

In [ ]:
print("="*100)
print("BEFORE/AFTER COMPARISON")
print("="*100)
print()

# Original cleaned dataset
print("BEFORE (cleaned_dataset):")
print(f"  Total samples: {initial_count:,}")
print(f"  Avg code length: {df['code_snippet'].str.len().mean():.0f} chars")
print(f"  Avg feedback length: {df['generated_feedback'].str.len().mean():.0f} chars")
print(f"  CoT feedbacks: {cot_count_before:,} ({cot_count_before/initial_count*100:.2f}%)")
print()

# After removing CoT
print("AFTER (no_cot_dataset):")
print(f"  Total samples: {len(df_clean):,}")
print(f"  Avg code length: {df_clean['code_snippet'].str.len().mean():.0f} chars")
print(f"  Avg feedback length: {df_clean['generated_feedback'].str.len().mean():.0f} chars")
print(f"  CoT feedbacks: 0 (0.00%)")
print()

# Impact
print("IMPACT:")
print(f"  Samples removed: {removed_count:,}")
print(f"  Retention rate: {len(df_clean)/initial_count*100:.2f}%")
print(f"  Avg feedback length change: {df_clean['generated_feedback'].str.len().mean() - df['generated_feedback'].str.len().mean():.0f} chars")

## 11. Save Clean Dataset (No CoT)

In [10]:
print("="*100)
print("SAVING CLEAN DATASET (NO COT)")
print("="*100)
print()

# Save as CSV
csv_output_path = Path('../data/cleaned_dataset_no_cot.csv')
df_clean.to_csv(csv_output_path, index=False)
print(f"✓ Saved to CSV: {csv_output_path}")
print(f"  Size: {csv_output_path.stat().st_size / 1024 / 1024:.2f} MB")

# Save as JSONL
jsonl_output_path = Path('../data/cleaned_dataset_no_cot.jsonl')
with open(jsonl_output_path, 'w', encoding='utf-8') as f:
    for _, row in df_clean.iterrows():
        json.dump(row.to_dict(), f, ensure_ascii=False)
        f.write('\n')
print(f"✓ Saved to JSONL: {jsonl_output_path}")
print(f"  Size: {jsonl_output_path.stat().st_size / 1024 / 1024:.2f} MB")

print()
print("✓ Dataset cleaning complete!")
print(f"  Final dataset: {len(df_clean):,} high-quality samples")

SAVING CLEAN DATASET (NO COT)

✓ Saved to CSV: ../data/cleaned_dataset_no_cot.csv
  Size: 6.07 MB
✓ Saved to JSONL: ../data/cleaned_dataset_no_cot.jsonl
  Size: 7.06 MB

✓ Dataset cleaning complete!
  Final dataset: 11,806 high-quality samples


## 12. Generate Removal Report

In [11]:
# Create a detailed report
report = f"""# CoT Feedback Removal Report

## Summary

- **Date**: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
- **Original dataset**: cleaned_dataset.csv
- **Output dataset**: cleaned_dataset_no_cot.csv

## Statistics

### Before Removal
- Total samples: {initial_count:,}
- Feedbacks with CoT: {cot_count_before:,} ({cot_count_before/initial_count*100:.2f}%)
- Clean feedbacks: {initial_count - cot_count_before:,} ({(initial_count - cot_count_before)/initial_count*100:.2f}%)

### After Removal
- Total samples: {len(df_clean):,}
- Samples removed: {removed_count:,} ({removed_count/initial_count*100:.2f}%)
- Retention rate: {len(df_clean)/initial_count*100:.2f}%

## Detection Patterns Used

A total of {len(COT_PATTERNS)} patterns were used to detect CoT feedbacks:

"""

for i, pattern in enumerate(COT_PATTERNS, 1):
    report += f"{i}. `{pattern}`\n"

report += f"\n## Most Common Patterns Found\n\n"

for pattern, count in sorted_patterns[:10]:
    percentage = (count / initial_count) * 100
    report += f"- `{pattern}`: {count:,} occurrences ({percentage:.2f}%)\n"

report += f"""\n## Quality Improvement

By removing feedbacks with CoT patterns, we ensure:
- **Direct feedback**: No meta-commentary or self-questioning
- **Clean training signal**: No internal reasoning artifacts
- **Consistent format**: All feedbacks follow the same structure
- **Better model quality**: Training on clean, focused feedback

## Output Files

- `cleaned_dataset_no_cot.csv`: Clean dataset in CSV format
- `cleaned_dataset_no_cot.jsonl`: Clean dataset in JSONL format

## Next Steps

Use `cleaned_dataset_no_cot.csv` for:
1. Training models (fine-tuning)
2. Pushing to Hugging Face Hub
3. Further analysis and evaluation
"""

# Save report
report_path = Path('../data/cot_removal_report.md')
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report)

print("="*100)
print("REPORT GENERATED")
print("="*100)
print()
print(f"✓ Report saved to: {report_path}")
print()
print(report)

REPORT GENERATED

✓ Report saved to: ../data/cot_removal_report.md

# CoT Feedback Removal Report

## Summary

- **Date**: 2025-12-03 14:05:24
- **Original dataset**: cleaned_dataset.csv
- **Output dataset**: cleaned_dataset_no_cot.csv

## Statistics

### Before Removal
- Total samples: 11,898
- Feedbacks with CoT: 92 (0.77%)
- Clean feedbacks: 11,806 (99.23%)

### After Removal
- Total samples: 11,806
- Samples removed: 92 (0.77%)
- Retention rate: 99.23%

## Detection Patterns Used

A total of 26 patterns were used to detect CoT feedbacks:

1. `Is your feedback`
2. `Does your feedback`
3. `Is the feedback`
4. `Does the feedback`
5. `Yes[,\s]+your feedback is`
6. `Yes[,\s]+the feedback is`
7. `No[,\s]+your feedback`
8. `No[,\s]+the feedback`
9. `neutral and do not contain`
10. `does not contain any positive or negative`
11. `without positive or negative connotation`
12. `allow the student to rapidly`
13. `allows the student to`
14. `help the student identify`
15. `helps the student`
1

## Summary

This notebook has:
1. ✓ Detected CoT patterns in feedbacks
2. ✓ Analyzed pattern frequency
3. ✓ Showed examples of problematic feedbacks
4. ✓ Removed all CoT feedbacks
5. ✓ Validated the clean dataset
6. ✓ Saved the clean dataset
7. ✓ Generated a detailed report

**Output**: `cleaned_dataset_no_cot.csv` - Ready for training! 🚀